# Multimodal Messages with LangChain

This reference notebook demonstrates how to send text, image, and audio content to LangChain agents using structured `HumanMessage` content blocks.

## Learning goals

- Configure an agent and send a text-only message.
- Upload an image, convert it to base64, and include it in a multimodal message.
- Record audio, encode it as base64 WAV data, and send it to an audio-capable model.
- Understand the relationship between a content block's `type`, encoded data, and MIME type.

## Before you run the notebook

1. Set the required model-provider API key in your environment or `.env` file.
2. Run the cells from top to bottom so variables such as `agent`, `img_b64`, and `aud_b64` exist before they are used.
3. The image section requires an uploaded image, and the audio section requires a working microphone and audio input device.

Each request follows the same pattern: create a `HumanMessage`, place it in the agent state under `messages`, invoke the agent, and print the final message content.

In [ ]:
# Load API keys and other local settings from the project's .env file.
from dotenv import load_dotenv

load_dotenv()

True

In [15]:
from langchain.agents import create_agent

# The system prompt establishes the agent's role and response style.
# gpt-5-nano is used for the text and image examples.
agent = create_agent(
    model="gpt-5-nano",
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [ ]:
from langchain.messages import HumanMessage

# A message content list allows text and other modalities to be sent together.
question = HumanMessage(
    content=[
        {"type": "text", "text": "What is the capital of The Moon?"}
    ]
)

# Agent state expects a list of messages under the "messages" key.
response = agent.invoke({"messages": [question]})

# The final message is the agent's answer after it processes the request.
print(response["messages"][-1].content)

In this science-fiction setting, the Moon’s capital is Selene Prime.

- Location: Perched on the rim of Shackleton Crater near the Moon’s south pole, catching sunlight at the edge of the crater while guarding shadowed ice caves below.
- Government: Seat of the Lunar Confederation, governed by a Prime Administrator and a Council of Craters who meet in the Assembly Hall.
- Cityscape: A ring of gravity-stabilized domes and glass towers ringed by vast solar concentrators. Habitats are pressurized, with transit tunnels threading the crater rim.
- Economy: Ice mining and water processing, lunar regolith fabrication, and solar energy exports; a hub for science, diplomacy, and tunneling infrastructure.
- Culture and daily life: A society adapted to long lunar days and nights, with community rituals around solar festivals and the «twilight markets» where Earth goods blend with local crafts.
- Landmarks: The Aperture Gate (a monumental solar mirror complex), the Glass Hemisphere (cultural distri

## Image Input

The image workflow has three steps: collect an image with a notebook widget, encode its bytes as base64 text, and attach that data to a `HumanMessage` alongside an instruction.

In [14]:
from ipywidgets import FileUpload
from IPython.display import display

# Restrict the widget to one image so the following cells have one clear input.
uploader = FileUpload(accept="image/*", multiple=False)
display(uploader)

FileUpload(value=(), accept='image/*', description='Upload')

In [ ]:
# Inspect the widget value to confirm that an image has been selected.
print(uploader.value)

({'name': 'panda_silhouette_on_matrix.png', 'type': 'image/png', 'size': 743106, 'content': <memory at 0x115af1e40>, 'last_modified': datetime.datetime(2026, 7, 22, 12, 17, 6, 288000, tzinfo=datetime.timezone.utc)},)


In [ ]:
import base64

if not uploader.value:
    raise ValueError(
        "No file uploaded yet. Click the 'Upload' button above, select an image, "
        "and re-run this cell."
    )

# FileUpload stores the selected file as a one-item tuple of dictionaries.
uploaded_file = uploader.value[0]

# The content field is a memoryview, so convert it to bytes before encoding it.
img_bytes = bytes(uploaded_file["content"])
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

# Preserve the uploaded MIME type so JPEG, PNG, and other image formats work.
image_mime_type = uploaded_file["type"]

In [ ]:
# Put the natural-language instruction and image data in the same message.
multimodal_question = HumanMessage(
    content=[
        {"type": "text", "text": "Tell me about this capital"},
        {
            "type": "image",
            "base64": img_b64,
            "mime_type": image_mime_type,
        },
    ]
)

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)

Here’s a vivid portrait of the capital suggested by your image—the neon horse carved in green code, a city that lives at the intersection of data and memory.

Name and symbol
- Equorium (the Mare Gate): The capital of the Confederation of Neon Domains. Its emblem is a luminous horse head formed from cascading green data streams, a symbol of speed, memory, and the city’s unbreakable link to information.

Geography and layout
- Built on a ring of floating districts tethered to a central arcology, Equorium floats above a quiet sea that glows with bioluminescent algae. The skyline is a lattice of iridescent spires and platform bridges, all connected by data-light roads that pulse with the city’s rhythm.
- The heart of the city sits on a broad plateau crowned by the Archive Spiral, a spiraling data hub that contains the world’s most persistent memories.

Governance and society
- Government runs on a hybrid model: a living AI known as The Steed coordinates the Council of Nodes, while human r

### Image section summary

The image bytes remain local until they are placed in the message. Base64 makes the binary data transportable as text, while `mime_type` tells the model how to interpret those bytes. The text block supplies the task; the image block supplies the evidence.

## Audio Input

The audio workflow records a short WAV clip, stores it in memory, base64-encodes it, and sends it to an audio-capable model. This cell requires microphone permissions and a functioning input device.

In [16]:
import base64
import io
import time

import sounddevice as sd
from scipy.io.wavfile import write
from tqdm import tqdm

In [ ]:
# Recording settings: mono audio at CD-quality sample rate.
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(
    int(duration * sample_rate),
    samplerate=sample_rate,
    channels=1,
)

# sd.rec starts asynchronously, so wait for the recording to finish.
# The progress bar is only visual feedback during the five-second capture.
for _ in tqdm(range(duration * 10), desc="Recording"):
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write the NumPy audio array to an in-memory WAV file.
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

# Models receive the binary WAV data as base64 text in the message block.
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.55it/s]


Done.


In [ ]:
# Use an audio-capable model for the audio content block.
agent = create_agent(model="gpt-audio")

multimodal_question = HumanMessage(
    content=[
        {"type": "text", "text": "Tell me about this audio file"},
        {
            "type": "audio",
            "base64": aud_b64,
            "mime_type": "audio/wav",
        },
    ]
)

response = agent.invoke({"messages": [multimodal_question]})
print(response["messages"][-1].content)

Sure, I can help analyze the audio. Let me start by examining the content, such as the speech, language, background sounds, or other notable features. I'll describe what I can identify. Let me process the audio now.


### Audio section summary

The recording is converted to a WAV byte stream before base64 encoding. The `audio` content block and `audio/wav` MIME type distinguish this payload from the image example, while the surrounding message structure stays the same.

## Conclusion and reusable pattern

This notebook uses one consistent pattern across modalities:

1. Prepare the input and identify its MIME type.
2. Encode binary data as base64 text when required by the content block.
3. Build a `HumanMessage` with a text instruction and one or more modality blocks.
4. Invoke the agent with `{"messages": [message]}`.
5. Read the final assistant response from `response["messages"][-1].content`.

For future experiments, keep the message structure stable and change only the model, content block, or instruction. Text and image requests use `gpt-5-nano` here; audio requests use `gpt-audio`. Model availability, supported modalities, API credentials, microphone permissions, and payload limits can vary by provider, so verify those requirements before adapting this notebook.